# Agentic Resume-Screening RAG: From Naive to Agentic

This notebook is a live, hands-on tour of Retrieval-Augmented Generation, built around a single realistic problem: helping a recruiter screen candidates against 25 real resumes pulled from an actual applicant pool. We will start with the simplest possible RAG pipeline, break it on purpose in front of you, and fix it step by step — adding **re-ranking**, **adaptive RAG**, and **corrective RAG** along the way, each introduced with a plain-English definition before we build it. We finish by wiring all of it into a self-correcting **agentic** system with LangGraph that decides for itself when and how to search. Everything runs on the OpenAI API only, uses plain LangChain and LangGraph, and is intentionally kept small enough to read top to bottom in one sitting.

**What you need:** an OpenAI API key, and the `resumes/` folder that ships next to this notebook. This notebook is set up to run inside the `uv`-managed environment that ships alongside it (`pyproject.toml` / `.venv`) — see the note at the very end of this section if you're opening it somewhere else.

## Part 0 — Setup

We only need a handful of libraries for this whole notebook: `langchain` and `langchain-openai` for the LLM / embeddings / tool-calling plumbing, `langgraph` for the agent graph we build later, `pypdf` to pull text out of the resume PDFs, and `python-dotenv` to read the API key. Notice what is *not* here — no vector database server, no reranker service, nothing beyond LangChain itself: everything runs in-process, which is exactly what you want for prototyping and for a masterclass. If you ran `uv sync` from this folder before opening the notebook (as recommended), all of this is already installed and the cell below is just a no-op safety net; if you're running elsewhere (e.g. Colab), swap it for `%pip install -qU langchain langchain-openai langgraph pypdf python-dotenv` instead.

In [1]:
!uv pip install -qU langchain langchain-openai langgraph pypdf python-dotenv

With the libraries in place, the next thing we need is your OpenAI API key. It lives in a local `.env` file next to this notebook — never pasted into a cell, never hard-coded, and listed in `.gitignore` so it can't accidentally get committed. `load_dotenv()` reads it into the environment automatically; if it's missing for any reason, the cell falls back to asking for it interactively instead.

In [2]:
import os
import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

Last piece of setup: two model names, kept as plain constants so you can swap them in one place. We use a small, cheap chat model for every LLM call in this notebook (there are quite a few, since our agent will call the model in a loop later on) and a small embedding model for vector search. If either name is retired on your account by the time you run this, just change the string here — nothing else in the notebook needs to know.

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

MODEL_NAME = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

## Part 1 — Loading the Resumes

We're working with 25 real, unmodified resumes pulled straight from the open-source [Resume-Screening-RAG-Pipeline](https://github.com/Hungreeee/Resume-Screening-RAG-Pipeline) dataset: seven Java developers, two PHP developers, two Hadoop developers, five business/systems analysts, and nine people in project-management, Scrum, and QA-adjacent roles (project managers, a scrum master, a technical program manager, a DevOps-leaning PM, QA/testing engineers). They deliberately overlap in places, the same way a real applicant pool would, so retrieval actually has to work to tell them apart. Nothing has been cleaned or reformatted; you'll see the same slightly messy PDF-to-text extraction a production system would have to deal with.

In [4]:
from pypdf import PdfReader

RESUME_DIR = "resumes"

CANDIDATES = [
    {"id": "mahesh_java", "file": "mahesh_java_developer.pdf", "name": "Mahesh", "role": "Sr. Full Stack Java Developer"},
    {"id": "sharath_java", "file": "sharath_java_developer.pdf", "name": "Sharath", "role": "Java Full Stack Developer"},
    {"id": "anudeep_java", "file": "anudeep_java_developer.pdf", "name": "Anudeep", "role": "Sr. Java Programmer"},
    {"id": "chetan_java", "file": "chetan_java_developer.pdf", "name": "Chetan Babu", "role": "Sr. Java Developer"},
    {"id": "pavan_java", "file": "pavan_java_developer.pdf", "name": "Pavan Kumar", "role": "Full Stack Java Developer"},
    {"id": "ashwini_java", "file": "ashwini_j2ee_developer.pdf", "name": "Ashwini", "role": "Sr. Java/J2EE Developer"},
    {"id": "sougandh_java", "file": "sougandh_java_developer.pdf", "name": "Sougandh", "role": "Java Fullstack Developer"},
    {"id": "nilesh_php", "file": "nilesh_php_lead.pdf", "name": "Nilesh Banthiya", "role": "PHP Lead / Web Developer"},
    {"id": "venkata_php", "file": "venkata_php_developer.pdf", "name": "Venkata", "role": "Sr. PHP / Drupal Developer"},
    {"id": "mani_hadoop", "file": "mani_hadoop_developer.pdf", "name": "Mani", "role": "Sr. Hadoop Developer"},
    {"id": "bapuji_hadoop", "file": "bapuji_hadoop_developer.pdf", "name": "Bapuji", "role": "Sr. Hadoop Developer"},
    {"id": "navneet_ba", "file": "navneet_business_analyst.pdf", "name": "Navneet", "role": "Business Analyst / Consultant"},
    {"id": "shail_ba", "file": "shail_business_analyst.pdf", "name": "Shail Tank", "role": "Business Analyst"},
    {"id": "krishna_bsa", "file": "krishna_business_systems_analyst.pdf", "name": "Krishna", "role": "Sr. Business Systems Analyst"},
    {"id": "amar_bsa", "file": "amar_business_systems_analyst.pdf", "name": "Amar", "role": "Business Systems Analyst"},
    {"id": "priyanka_bsa", "file": "priyanka_business_systems_analyst.pdf", "name": "Priyanka", "role": "Sr. Business Systems Analyst"},
    {"id": "avinash_pm", "file": "avinash_project_manager.pdf", "name": "Avinash", "role": "Project Manager"},
    {"id": "srivatsan_pm", "file": "srivatsan_project_manager.pdf", "name": "Srivatsan Ramabhadran", "role": "Project Manager / Business Data Analyst"},
    {"id": "adelina_pm", "file": "adelina_project_manager.pdf", "name": "Adelina Erimia", "role": "Project Manager (PMP)"},
    {"id": "adhi_scrum", "file": "adhi_scrum_master.pdf", "name": "Adhi Gopalam", "role": "Certified Scrum Master & Business Analyst"},
    {"id": "ranjan_pm_scrum", "file": "ranjan_pm_scrum_master.pdf", "name": "Ranjan Kumar Verma", "role": "Project Manager / Scrum Master"},
    {"id": "raju_tpm", "file": "raju_technical_program_manager.pdf", "name": "Raju Goduguchinta", "role": "Sr. Technical Program Manager"},
    {"id": "ravi_pm_devops", "file": "ravi_pm_devops.pdf", "name": "Ravi Prasad Burra", "role": "Certified PM / DevOps-Automation Architect"},
    {"id": "murali_pm_qa", "file": "murali_pm_qa.pdf", "name": "Muralidhar Chandrashekar", "role": "Project Manager (QA/Testing)"},
    {"id": "gautami_qa", "file": "gautami_qa_mobile_testing.pdf", "name": "Gautami Bulusu", "role": "QA / Mobile Test Engineer"},
]

def extract_pdf_text(path):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

full_resumes = {}
for c in CANDIDATES:
    text = extract_pdf_text(os.path.join(RESUME_DIR, c["file"]))
    full_resumes[c["id"]] = {"name": c["name"], "role": c["role"], "text": text}

for cid, info in full_resumes.items():
    print(f"{cid:16s} | {info['name']:24s} | {info['role']:44s} | {len(info['text']):5d} chars")

mahesh_java      | Mahesh                   | Sr. Full Stack Java Developer                | 25761 chars
sharath_java     | Sharath                  | Java Full Stack Developer                    | 14013 chars
anudeep_java     | Anudeep                  | Sr. Java Programmer                          | 14168 chars
chetan_java      | Chetan Babu              | Sr. Java Developer                           | 15102 chars
pavan_java       | Pavan Kumar              | Full Stack Java Developer                    | 32349 chars
ashwini_java     | Ashwini                  | Sr. Java/J2EE Developer                      | 21173 chars
sougandh_java    | Sougandh                 | Java Fullstack Developer                     | 28724 chars
nilesh_php       | Nilesh Banthiya          | PHP Lead / Web Developer                     |  3422 chars
venkata_php      | Venkata                  | Sr. PHP / Drupal Developer                   | 11855 chars
mani_hadoop      | Mani                     | Sr. Hadoo

Twenty-five candidates, ranging from resumes of a few thousand characters to ones well over 30,000. That size difference matters more than it looks — it's exactly what will cause problems once we start chunking text for retrieval, which is where we're headed next.

## Part 2 — A Basic RAG Pipeline

The standard recipe: split every resume into overlapping chunks, embed each chunk, and drop them all into a vector store. We use LangChain's `RecursiveCharacterTextSplitter` with an 800-character chunk size — small enough that a short resume breaks into just 5 pieces and our longest one breaks into over 50 — and an in-memory vector store built into `langchain-core`, so there's no external database to install or run for this class.

One practical note before we run this: splitting text into chunks is free, it's just Python string manipulation. *Embedding* those chunks is not — it's one OpenAI API call per chunk, and with 25 resumes that's several hundred chunks. If you re-run this notebook a few times while prepping a class, or restart the kernel mid-session, you don't want to pay for (and wait on) the same embeddings again. So instead of embedding on every run, we'll embed once and cache the whole vector store — chunks *and* their embedding vectors — to a local JSON file next to the notebook. Every run after the first just loads that file back into memory; zero embedding calls, and it's near-instant.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

CACHE_PATH = "cache/resume_vectorstore.json"

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)

def build_chunk_documents():
    docs = []
    for cid, info in full_resumes.items():
        for i, chunk in enumerate(splitter.split_text(info["text"])):
            docs.append(
                Document(
                    page_content=chunk,
                    metadata={"candidate_id": cid, "name": info["name"], "role": info["role"], "chunk_index": i},
                )
            )
    return docs

def load_or_build_vectorstore():
    expected_ids = sorted(full_resumes.keys())

    if os.path.exists(CACHE_PATH):
        store = InMemoryVectorStore.load(CACHE_PATH, embeddings)
        cached_ids = sorted({row["metadata"]["candidate_id"] for row in store.store.values()})
        if cached_ids == expected_ids:
            print(f"Loaded {len(store.store)} chunks (with embeddings already attached) from cache - 0 API calls made.")
            return store
        print("Cache doesn't match the current candidate list - rebuilding and re-embedding...")

    chunk_documents = build_chunk_documents()
    print(f"No usable cache found - embedding {len(chunk_documents)} chunks via the OpenAI API (one-time cost)...")
    store = InMemoryVectorStore.from_documents(chunk_documents, embeddings)
    store.dump(CACHE_PATH)
    print(f"Cached chunks + embeddings to {CACHE_PATH} for next time.")
    return store

vectorstore = load_or_build_vectorstore()

Loaded 666 chunks (with embeddings already attached) from cache - 0 API calls made.


With the chunks embedded, a "basic RAG pipeline" is just three steps repeated on every question: retrieve the top-k most similar chunks, stuff them into a prompt, and ask the model to answer using only that context. The helper below does exactly that, and also prints which chunks it retrieved so we can see the retriever's work, not just the model's — that transparency is going to matter a lot in the next section.

In [6]:
NAIVE_SYSTEM_PROMPT = (
    "You are a hiring assistant. Answer the recruiter's question using only the "
    "resume excerpts provided as context."
)

def naive_rag(query, k=4):
    hits = vectorstore.similarity_search(query, k=k)
    print("Retrieved chunks:")
    for h in hits:
        print(f"  - {h.metadata['name']} (chunk #{h.metadata['chunk_index']})")

    context = "\n\n".join(f"[{h.metadata['name']}]\n{h.page_content}" for h in hits)
    prompt = f"{NAIVE_SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}"
    answer = llm.invoke(prompt).content

    print("\nAnswer:\n" + answer)
    return hits, answer

Let's try it on the kind of question this pipeline is built for: matching a role to a candidate. This is the easy case, deliberately — a single, well-defined skill set that should map cleanly onto one obviously-relevant resume.

In [7]:
_ = naive_rag(
    "We're hiring a senior backend engineer with strong Java/J2EE experience "
    "and exposure to cloud platforms like AWS. Who would you recommend and why?"
)

Retrieved chunks:
  - Chetan Babu (chunk #4)
  - Sharath (chunk #5)
  - Mani (chunk #3)
  - Pavan Kumar (chunk #17)



Answer:
I would recommend Sharath for the senior backend engineer position. He has extensive experience as a Full Stack Java Developer, specifically working with J2EE patterns, which aligns well with the requirements for strong Java/J2EE experience. Additionally, he has hands-on experience with AWS services such as EC2, S3, and RDS, demonstrating his exposure to cloud platforms. His background in gathering requirements and interacting with clients also indicates strong communication skills, which are essential for a senior role.


## Part 3 — Where This Breaks

That worked because the question happened to line up with how the chunks got embedded. Real recruiters don't ask questions that considerately — they ask about specific people, they ask for comparisons across the whole pool, and they ask things resumes simply don't contain. Let's push on the same pipeline with three realistic questions and watch it struggle.

**Problem 1: fragmentation.** Mahesh's resume is ~26,000 characters long, which our 800-character chunker turned into 36 separate pieces. His total years of experience is stated once, in the summary near the top; a specific healthcare client he worked for is mentioned only in a project entry many chunks later. A question that needs both facts is a question that needs two chunks that are far apart in the document — and nothing about similarity search guarantees the retriever grabs both.

In [8]:
FRAGMENTATION_QUERY = "How many years of experience does Mahesh have in total, and which healthcare client did he work for?"
_ = naive_rag(FRAGMENTATION_QUERY)

Retrieved chunks:
  - Mahesh (chunk #0)
  - Bapuji (chunk #5)
  - Sougandh (chunk #37)
  - Muralidhar Chandrashekar (chunk #25)



Answer:
Mahesh has over 8 years of experience in software development. He worked for a healthcare client, specifically on a comprehensive and integrated Hospital Management System for a Super Specialty Hospital while at Systems & Services Limited in Hyderabad, India.


Look at the printed chunk list above: were both facts actually retrieved together, or did the model have to guess, hedge, or quietly drop one of them? Rerun the cell above a few times, or lower `k`, and you'll see this is not a one-off — it's a structural consequence of chunking a document and then retrieving pieces of it independently.

**Problem 2: coverage.** Now ask something that needs a wide slice of the pool, not just the single closest match. With `k=4` chunks pulled from a pool of several hundred spread across 25 people, there is no way the retriever can represent everyone relevant — and worse, similarity search has no notion of "one chunk per person," so it can easily return four chunks from only two or three resumes. We can prove this without even looking at the model's answer, just by checking which candidates actually made it into the context.

In [9]:
RUNNING_QUERY = (
    'Shortlist every candidate in the pool with project-management or Scrum-master '
    'experience for a "Technical Delivery Lead" panel review, and give evidence for each one you include.'
)

hits, answer = naive_rag(RUNNING_QUERY)

covered = {h.metadata["candidate_id"] for h in hits}
missing = [c["name"] for c in CANDIDATES if c["id"] not in covered]
print(f"\nCandidates represented in context: {len(covered)} / {len(full_resumes)}")
print("Missing entirely:", missing)

Retrieved chunks:
  - Ravi Prasad Burra (chunk #11)
  - Avinash (chunk #2)
  - Srivatsan Ramabhadran (chunk #3)
  - Pavan Kumar (chunk #7)



Answer:
Based on the provided resume excerpts, the following candidates have project management or Scrum Master experience suitable for a "Technical Delivery Lead" panel review:

1. **Ravi Prasad Burra**
   - Evidence: 
     - Expertized in Project Reporting & estimation using MS-Project/Excel.
     - Experienced in defining Test Strategy and approaches.
     - Facilitated all scrum ceremonies such as grooming, sprint planning, retrospectives, and daily stand-ups.
     - Empowered teams to self-organize and grow cross-functionality, indicating strong leadership and Scrum Master capabilities.

2. **Srivatsan Ramabhadran**
   - Evidence:
     - Demonstrates excellent management and leadership qualities, acting as the go-to person on project-related issues.
     - Hands-on experience in various project management models, including Agile, and performed GAP analysis to identify problems.
     - Strong knowledge of project management skills such as time estimation, task identification, risk

The answer above may sound perfectly confident about the whole shortlist — but the printed coverage count tells the real story. Whatever candidates are missing from that context were shortlisted, if they were mentioned at all, purely from the model's imagination. This is exactly the kind of failure that's invisible unless you go looking for it, which is why we'll keep printing retrieval diagnostics for the rest of the notebook.

**Problem 3: answering questions the resumes can't answer.** None of these documents mention salary expectations or relocation preferences — that information was never collected. A trustworthy system should say so plainly. Let's see what a plain "stuff the context and ask" pipeline does instead.

In [10]:
_ = naive_rag("What is Mahesh's salary expectation, and would he be willing to relocate to another city?")

Retrieved chunks:
  - Mahesh (chunk #0)
  - Muralidhar Chandrashekar (chunk #25)
  - Muralidhar Chandrashekar (chunk #2)
  - Srivatsan Ramabhadran (chunk #39)



Answer:
The provided resume excerpts do not include any information regarding Mahesh's salary expectation or his willingness to relocate to another city.


Three real problems, all from a pipeline that looks completely reasonable on paper: **fragmentation** (a document's own information gets scattered across chunks), **coverage** (a fixed top-k can silently exclude entire candidates), and **grounding** (nothing stops the model from answering questions the data can't actually answer). We'll fix fragmentation and grounding directly in the next two parts. Coverage is the hardest of the three — no amount of clever chunking or retrieval-quality tuning fixes it on its own — and it's what motivates everything from re-ranking onward, all the way to the agent at the end.

## Part 4 — Fixing Fragmentation: Small-to-Big Retrieval

The fix for fragmentation doesn't require better chunking or a bigger model — it requires *searching* at the chunk level (precise) while *answering* at the document level (complete). This is called small-to-big retrieval, and it's actually the exact technique the original open-source project this dataset comes from uses in production: find the most relevant small pieces, then trace each one back to its full source document before handing anything to the LLM.

In [11]:
def search_full_resumes(query, k=4):
    hits = vectorstore.similarity_search(query, k=k)
    seen_ids = list(dict.fromkeys(h.metadata["candidate_id"] for h in hits))
    print("Candidates retrieved (chunk hit -> full resume):", [full_resumes[cid]["name"] for cid in seen_ids])

    return "\n\n".join(
        f"--- {full_resumes[cid]['name']} ({full_resumes[cid]['role']}) ---\n{full_resumes[cid]['text']}"
        for cid in seen_ids
    )

`search_full_resumes` still uses the same chunk-level vector search we already built — that part was never the problem — it just swaps each chunk's text for the *whole* resume before it goes anywhere near the LLM. Let's route the fragmentation question through this instead and compare.

In [12]:
def grounded_rag(query, k=4):
    context = search_full_resumes(query, k=k)
    prompt = f"{NAIVE_SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}"
    answer = llm.invoke(prompt).content
    print(f"\n(Context given to the model: {len(context)} characters)")
    print("\nAnswer:\n" + answer)
    return context, answer

_ = grounded_rag(FRAGMENTATION_QUERY)

Candidates retrieved (chunk hit -> full resume): ['Mahesh', 'Bapuji', 'Sougandh', 'Muralidhar Chandrashekar']



(Context given to the model: 88653 characters)

Answer:
Mahesh has over 8 years of experience in total. He worked for the healthcare client ASD Health Care in Frisco, TX.


Mahesh's full resume is now in the prompt, so both the years-of-experience figure and the healthcare client are sitting right next to each other for the model to read — fragmentation, for this candidate at least, is gone. Notice what we did *not* have to do: no bigger chunks, no fancier embedding model, just a different mapping from search results to context.

## Part 5 — Fixing Trust: Structured, Grounded Output

Small-to-big retrieval fixes what the model *sees*; it does nothing about what the model is willing to *say* when the answer isn't in there. For that we lean on structured output: instead of free text, we ask the model to fill in a Pydantic schema that requires a citation and, critically, a field for anything the question asked about that the resume simply doesn't cover. A model that has to fill in an "unknowns" field is a lot less likely to quietly invent a salary figure.

In [13]:
from pydantic import BaseModel, Field

class CandidateAssessment(BaseModel):
    candidate_id: str = Field(description="The candidate's id exactly as given in the context")
    name: str
    match_score: int = Field(description="0-10 fit score for the role, 0 = not relevant, 10 = excellent fit")
    evidence: str = Field(description="A short quote or close paraphrase from the resume backing up the score")
    unknowns: str = Field(description="Anything the question asked about that is NOT stated in the resume, or 'none'")

class ScreeningReport(BaseModel):
    summary: str = Field(description="A 1-2 sentence overview of the recommendation")
    candidates: list[CandidateAssessment]

structured_llm = llm.with_structured_output(ScreeningReport)

With the schema defined, `screen_candidates` is almost the same function as `grounded_rag` — same retrieval, same context — just pointed at `structured_llm` instead, with an explicit instruction to use the `unknowns` field rather than guess. Let's ask the salary/relocation question again and see the difference.

In [14]:
def screen_candidates(query, k=4):
    context = search_full_resumes(query, k=k)
    prompt = (
        f"{NAIVE_SYSTEM_PROMPT} Only use facts present in the context. If the question asks "
        f"about something not stated in a resume, say so in 'unknowns' instead of guessing.\n\n"
        f"Context:\n{context}\n\nQuestion: {query}"
    )
    return structured_llm.invoke(prompt)

report = screen_candidates("What is Mahesh's salary expectation, and would he be willing to relocate to another city?")
print(report.summary, "\n")
for c in report.candidates:
    print(f"{c.name:10s} score={c.match_score}/10")
    print(f"  evidence: {c.evidence}")
    print(f"  unknowns: {c.unknowns}\n")

Candidates retrieved (chunk hit -> full resume): ['Mahesh', 'Muralidhar Chandrashekar', 'Srivatsan Ramabhadran']


The resume does not provide any information regarding Mahesh's salary expectations or his willingness to relocate. 

Mahesh     score=0/10
  evidence: None
  unknowns: Salary expectation and willingness to relocate are not stated in the resume.



That's a genuinely more trustworthy answer — every score is tied to a quote, and anything ungrounded is labeled instead of invented. But structure doesn't fix coverage: run the exact same "shortlist every PM/Scrum candidate" question from Part 3 through this pipeline, and check how many candidates actually show up.

In [15]:
report = screen_candidates(RUNNING_QUERY)
covered_names = sorted(c.name for c in report.candidates)
print(f"Candidates assessed: {len(covered_names)} / {len(full_resumes)} -> {covered_names}")

Candidates retrieved (chunk hit -> full resume): ['Ravi Prasad Burra', 'Avinash', 'Srivatsan Ramabhadran', 'Pavan Kumar']


Candidates assessed: 2 / 25 -> ['Avinash', 'Ravi Prasad Burra']


Still incomplete — grounding a wrong-sized context just gives you an honest, well-cited, *incomplete* answer instead of a dishonest one. Coverage is a retrieval-strategy problem, not a prompting problem, and fixing it properly takes a few more ideas: sharper ranking of what we retrieve, a system that adapts its strategy to the kind of question being asked, and a system that checks its own retrieval and corrects course when it comes up short. We'll build all three next, and then hand the resulting toolkit to an agent that can decide, on its own, how and when to use each one.

## Part 6 — Re-ranking: Fixing Precision, Not Just Coverage

**Definition.** Vector similarity search is fast because it reduces "is this chunk relevant?" to a single number: the cosine distance between two embeddings. That's a *proxy* for relevance, not relevance itself, and proxies break down under pressure — the more chunks in the pool, and the more similar candidates are to each other, the more often "closest in embedding space" and "actually answers the question" quietly diverge. **Re-ranking** is a second, more careful pass that fixes this: over-fetch a wider set of candidates with cheap vector search (say k=15 instead of k=4), then re-score every one of them with something slower but far more accurate, and keep only the best few. It's a standard two-stage pattern in production search systems — cast a wide net cheaply, then spend your expensive compute only on the shortlist.

**Example.** In a moment we'll ask *"which candidate has hands-on experience with automated testing frameworks like Selenium?"* One of our candidates, Sougandh, has a resume chunk that says almost exactly that: *"Expertise in Selenium automation using Selenium WebDriver, Selenium Grid, JAVA, JUnit..."* — about as direct a match as a chunk can be. Plain top-4 vector search doesn't return it. Instead it slips in a chunk from Mahesh, a Java developer, that only talks about general CI/CD deployment and never says "Selenium" at all — it just happens to sit close enough in embedding space, because "testing frameworks" and "CI/CD deployment" both live in similar technical-vocabulary territory. A reranker reads both chunks against the literal question and fixes the ordering.

The re-ranker most production systems reach for is a *cross-encoder* — a small model trained specifically to score (query, passage) pairs together, rather than embedding them separately. We don't have one of those installed (staying OpenAI-only, remember), so we'll build the same idea using the chat model itself as the judge: hand it the query and every over-fetched chunk in one batch, and ask it to score each one's relevance. One LLM call, no matter how many chunks we're reranking — that keeps the cost and latency predictable.

In [16]:
class ChunkScore(BaseModel):
    index: int = Field(description="The index of the chunk being scored, exactly as given")
    score: int = Field(description="0-10 relevance of this chunk to the query, 10 = directly answers it")

class RerankResult(BaseModel):
    scores: list[ChunkScore]

reranker_llm = llm.with_structured_output(RerankResult)

def rerank(query, chunks, top_n=4):
    listing = "\n\n".join(f"[{i}] ({c.metadata['name']}): {c.page_content}" for i, c in enumerate(chunks))
    result = reranker_llm.invoke(
        f"Query: {query}\n\nScore how relevant each numbered chunk below is to answering the query, 0-10. "
        f"Judge the actual content, not just topical similarity.\n\n{listing}"
    )
    score_by_index = {s.index: s.score for s in result.scores}
    ranked = sorted(enumerate(chunks), key=lambda pair: score_by_index.get(pair[0], 0), reverse=True)
    return [chunk for _, chunk in ranked[:top_n]]

`rerank` takes whatever chunks a vector search already found and reorders them by actual relevance, keeping the top `top_n`. Let's see it earn its keep on the Selenium example above: compare plain top-4 vector search against "over-fetch 15, then rerank down to 4."

In [17]:
RERANK_QUERY = "Which candidate has hands-on experience with automated testing frameworks like Selenium?"

naive_hits = vectorstore.similarity_search(RERANK_QUERY, k=4)
print("Naive top-4 (vector similarity only):", [h.metadata["name"] for h in naive_hits])

overfetched = vectorstore.similarity_search(RERANK_QUERY, k=15)
reranked_hits = rerank(RERANK_QUERY, overfetched, top_n=4)
print("Reranked top-4 (over-fetch 15, then LLM rerank):", [h.metadata["name"] for h in reranked_hits])

Naive top-4 (vector similarity only): ['Gautami Bulusu', 'Ravi Prasad Burra', 'Gautami Bulusu', 'Mahesh']


Reranked top-4 (over-fetch 15, then LLM rerank): ['Gautami Bulusu', 'Gautami Bulusu', 'Sougandh', 'Ravi Prasad Burra']


Compare the two lists above — if reranking is doing real work, Sougandh should now show up in the reranked list even though plain vector search missed him entirely, because his chunk explicitly names Selenium while the chunks vector search preferred only gestured at "testing" and "CI/CD" in more general terms. (LLM judgments aren't perfectly deterministic even at temperature 0, so the exact top-4 can shift slightly between runs — the reliable signal is *which* candidates newly appear, not the precise ordering.) That's the whole value of the extra pass: not finding new information, just correctly prioritizing information that was already sitting there in the pool. Now let's fold this into the small-to-big retrieval we built in Part 4, so every future full-resume lookup in this notebook benefits from it: over-fetch chunks cheaply, rerank them for precision, *then* trace the survivors back to their full resumes.

In [18]:
def search_full_resumes_reranked(query, k=4, fetch_k=15):
    overfetched = vectorstore.similarity_search(query, k=fetch_k)
    top_chunks = rerank(query, overfetched, top_n=k)
    seen_ids = list(dict.fromkeys(c.metadata["candidate_id"] for c in top_chunks))
    print("Reranked candidates (chunk hit -> full resume):", [full_resumes[cid]["name"] for cid in seen_ids])

    return "\n\n".join(
        f"--- {full_resumes[cid]['name']} ({full_resumes[cid]['role']}) ---\n{full_resumes[cid]['text']}"
        for cid in seen_ids
    )